# Phase 1 – Data Preprocessing & Exploration

This notebook covers the entire Phase 1 pipeline:
1. Load and explore raw annotations (parsed from XML).
2. Visualise label distributions and pedestrian statistics.
3. Generate fixed-length sequences from pedestrian tracks.
4. Split sequences into train / validation / test sets (pedestrian-stratified).
5. Verify a sample frame can be loaded successfully.
6. Preview an occluded frame generated in Phase 2.

In [ ]:
# ──────────────────────────────────────────────
# Imports & Setup
# ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import cv2
import sys

sys.path.insert(0, str(Path.cwd().parent))  # project root

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

print("Setup complete ✓")

In [ ]:
# ──────────────────────────────────────────────
# Paths — actual data lives under datasets/
# ──────────────────────────────────────────────
DATA_ROOT = Path("../datasets")  # relative to notebooks/
METADATA_DIR = DATA_ROOT / "processed" / "metadata"
FRAMES_DIR    = DATA_ROOT / "processed" / "frames"
OCCLUDED_DIR  = DATA_ROOT / "processed" / "occluded_frames"

ANNOTATIONS_CSV = METADATA_DIR / "annotations.csv"
SEQUENCES_CSV   = METADATA_DIR / "sequences.csv"
TRAIN_CSV       = METADATA_DIR / "train.csv"
VAL_CSV         = METADATA_DIR / "val.csv"
TEST_CSV        = METADATA_DIR / "test.csv"

print(f"Metadata   → {METADATA_DIR}")
print(f"Frames     → {FRAMES_DIR}")
print(f"Occluded   → {OCCLUDED_DIR}")

---
## 1. Load Annotations

In [ ]:
# Load parsed annotations
df = pd.read_csv(ANNOTATIONS_CSV)
print(f"Rows : {len(df):,}")
print(f"Cols : {df.shape[1]}")
df.head(3)

In [ ]:
# Column info & missing values
print(df.info())
print()
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Descriptive stats for numeric fields
df.describe()

---
## 2. Label Distribution (Crossing vs Non-Crossing)

In [ ]:
# Value counts for the 'cross' label
label_counts = df["cross"].value_counts()
print("Label distribution:\n")
print(label_counts)

# Bar chart + Pie chart
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#2ecc71', '#e74c3c', '#95a5a6']

label_counts.plot(kind='bar', ax=ax[0], color=colors[:len(label_counts)], edgecolor='black')
ax[0].set_title('Crossing Label Counts')
ax[0].set_xlabel('Label')
ax[0].set_ylabel('Count')
ax[0].tick_params(axis='x', rotation=45)
for p in ax[0].patches:
    ax[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2., p.get_height()),
                   ha='center', va='bottom', fontsize=9)

label_counts.plot(kind='pie', ax=ax[1], autopct='%1.1f%%', colors=colors[:len(label_counts)],
                  startangle=90, wedgeprops={'edgecolor': 'black'})
ax[1].set_ylabel('')
ax[1].set_title('Crossing Label Proportions')

plt.tight_layout()
plt.show()

In [ ]:
# Remove irrelevant rows for later steps
df_clean = df[df["cross"] != "crossing-irrelevant"].copy()
print(f"Rows after removing 'crossing-irrelevant': {len(df_clean):,}")
print(f"Removed {len(df) - len(df_clean):,} rows")

---
## 3. Pedestrian Statistics

In [ ]:
unique_peds = df_clean["id"].nunique()
print(f"Unique pedestrians: {unique_peds}")

# Tracks per pedestrian
peds_counts = df_clean["id"].value_counts()
print(f"\nTrack length stats (frames per pedestrian):")
print(peds_counts.describe())

# Histogram
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.hist(peds_counts, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
ax.set_title('Distribution of Track Lengths per Pedestrian')
ax.set_xlabel('Number of frames')
ax.set_ylabel('Number of pedestrians')
ax.axvline(peds_counts.median(), color='red', linestyle='--',
           label=f'Median: {peds_counts.median():.0f}')
ax.axvline(peds_counts.mean(), color='green', linestyle='--',
           label=f'Mean: {peds_counts.mean():.1f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Videos in the dataset
videos = df_clean["video"].unique()
print(f"Number of videos: {len(videos)}")
print("Video IDs:", sorted(videos)[:10], "..." if len(videos) > 10 else "")

---
## 4. Sequence Generation

Group annotations by `(video, pedestrian_id)`, slide a fixed-width window (30 frames, stride 15) along each track, and label each sequence with the **last** frame's crossing status.

In [ ]:
# ── Configuration ──
WINDOW = 30
STRIDE = 15

# Sort
df_sorted = df_clean.sort_values(["video", "id", "frame"])

# Generate sequences
sequences = []
seq_id = 0

for (video, pid), group in df_sorted.groupby(["video", "id"]):
    group = group.reset_index(drop=True)
    if len(group) < WINDOW:
        continue
    for start in range(0, len(group) - WINDOW + 1, STRIDE):
        window = group.iloc[start:start + WINDOW]
        sequences.append({
            "sequence_id": seq_id,
            "video": video,
            "pedestrian_id": pid,
            "start_frame": int(window.iloc[0]["frame"]),
            "end_frame": int(window.iloc[-1]["frame"]),
            "frames": "|".join(window["frame"].astype(str).tolist()),
            "label": window.iloc[-1]["cross"]
        })
        seq_id += 1

seq_df = pd.DataFrame(sequences)
print(f"Total sequences generated: {len(seq_df):,}")
print(f"\nLabel distribution in sequences:")
print(seq_df["label"].value_counts())

In [ ]:
# Visualise sequence label distribution
label_seq_counts = seq_df["label"].value_counts()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

label_seq_counts.plot(kind='bar', ax=ax[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
ax[0].set_title('Sequence Label Counts')
ax[0].set_xlabel('Label')
ax[0].set_ylabel('Count')
for p in ax[0].patches:
    ax[0].annotate(f'{int(p.get_height()):,}',
                   (p.get_x() + p.get_width()/2., p.get_height()),
                   ha='center', va='bottom', fontsize=9)

label_seq_counts.plot(kind='pie', ax=ax[1], autopct='%1.1f%%',
                      colors=['#2ecc71', '#e74c3c'],
                      startangle=90, wedgeprops={'edgecolor': 'black'})
ax[1].set_ylabel('')
ax[1].set_title('Sequence Label Proportions')

plt.tight_layout()
plt.show()

In [ ]:
# Save sequences to disk
METADATA_DIR.mkdir(parents=True, exist_ok=True)
seq_df.to_csv(SEQUENCES_CSV, index=False)
print(f"Saved: {SEQUENCES_CSV}")

---
## 5. Dataset Splitting (Pedestrian-Stratified)

Split pedestrians into **70% train**, **15% validation**, **15% test**, then assign all sequences belonging to each pedestrian to the respective split.

In [ ]:
from sklearn.model_selection import train_test_split

# Unique pedestrians
pedestrians = seq_df["pedestrian_id"].unique()
print(f"Total unique pedestrians in sequences: {len(pedestrians)}")

# 70 / 15 / 15 split
train_peds, temp_peds = train_test_split(pedestrians, test_size=0.30, random_state=42)
val_peds, test_peds   = train_test_split(temp_peds, test_size=0.50, random_state=42)

train_df = seq_df[seq_df["pedestrian_id"].isin(train_peds)]
val_df   = seq_df[seq_df["pedestrian_id"].isin(val_peds)]
test_df  = seq_df[seq_df["pedestrian_id"].isin(test_peds)]

print(f"\n{'Split':<12} {'Sequences':<12} {'Pedestrians':<12}")
print('-' * 36)
print(f"{'Train':<12} {len(train_df):<12,} {len(train_peds):<12}")
print(f"{'Val':<12} {len(val_df):<12,} {len(val_peds):<12}")
print(f"{'Test':<12} {len(test_df):<12,} {len(test_peds):<12}")
print(f"\nCheck sum: {len(train_df) + len(val_df) + len(test_df)} (should equal {len(seq_df)})")

In [ ]:
# Bar chart for split sizes
split_sizes = [len(train_df), len(val_df), len(test_df)]
split_names = ['Train', 'Validation', 'Test']
split_colors = ['#2ecc71', '#f39c12', '#e74c3c']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(split_names, split_sizes, color=split_colors, edgecolor='black', width=0.5)
ax.set_title('Number of Sequences per Split')
ax.set_ylabel('Sequences')

for bar, size in zip(bars, split_sizes):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
            f'{size:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Save splits
train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV, index=False)
test_df.to_csv(TEST_CSV, index=False)
print(f"Train CSV  → {TRAIN_CSV}")
print(f"Val   CSV  → {VAL_CSV}")
print(f"Test  CSV  → {TEST_CSV}")

---
## 6. Frame Loading Verification

Use `ImageLoader` from `utils/` to load a sample frame and display it with the pedestrian bounding box overlaid.

In [ ]:
from utils.image_loader import ImageLoader

# Use the absolute frames path
loader = ImageLoader(str(FRAMES_DIR))

# Pick the first training sample
sample = train_df.iloc[0]
video  = sample["video"]
ped_id = sample["pedestrian_id"]
start  = sample["start_frame"]
label  = sample["label"]

print(f"Sample sequence -> video={video}, pedestrian={ped_id}, start_frame={start}, label={label}")

# Load the first frame of that sequence
img = loader.load_frame(video, start, set_name="set01")
print(f"Image shape: {img.shape}")

In [ ]:
# Retrieve bounding box for that pedestrian at that frame
bbox_row = df_clean[
    (df_clean["video"] == video) &
    (df_clean["id"] == ped_id) &
    (df_clean["frame"] == start)
]

if len(bbox_row) > 0:
    b = bbox_row.iloc[0]
    x1, y1, x2, y2 = int(b["x1"]), int(b["y1"]), int(b["x2"]), int(b["y2"])

    # Draw rectangle on image
    img_copy = img.copy()
    cv2.rectangle(img_copy, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(img_copy, f"{label} (ID:{ped_id})", (x1, max(y1-5, 20)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display (OpenCV loads BGR, convert to RGB for matplotlib)
    img_rgb = cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    ax.imshow(img_rgb)
    ax.set_title(f"Video: {video} | Frame: {start} | Pedestrian: {ped_id} | Label: {label}")
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Bounding box not found for this frame.")

---
## 7. Occlusion Preview (Phase 2)

Sample an occluded frame generated by `scripts/phase2/generate_occlusions.py` and compare it side-by-side with the original.

In [ ]:
# Define occlusion levels
OCCLUSION_LEVELS = ["occ25", "occ50", "occ75"]
OCC_TEST_DIR = OCCLUDED_DIR / "test"

if OCC_TEST_DIR.exists():
    # Load original frame (same one used in occlusion generation: video_0001, frame 1013)
    occ_video = "video_0001"
    occ_frame = 1013

    try:
        original_img = loader.load_frame(occ_video, occ_frame, set_name="set01")
        original_rgb = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))

        # Original
        axes[0].imshow(original_rgb)
        axes[0].set_title(f"Original\n{occ_video} frame {occ_frame}")
        axes[0].axis('off')

        # Each occlusion level
        for i, level in enumerate(OCCLUSION_LEVELS):
            occ_path = OCC_TEST_DIR / f"{occ_frame}_{level}.jpg"
            if occ_path.exists():
                occ_img = cv2.imread(str(occ_path))
                occ_rgb = cv2.cvtColor(occ_img, cv2.COLOR_BGR2RGB)
                axes[i + 1].imshow(occ_rgb)
                axes[i + 1].set_title(f"Occlusion {level}")
            else:
                axes[i + 1].text(0.5, 0.5, "Not found", ha='center', va='center')
            axes[i + 1].axis('off')

        plt.suptitle("Original vs. Occluded Frames (Phase 2)", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    except FileNotFoundError as e:
        print("Original frame not found:", e)
else:
    print(f"Occluded directory not found: {OCC_TEST_DIR}")
    print("Run scripts/phase2/generate_occlusions.py first to generate occlusion samples.")

---
## 8. Summary Table

In [ ]:
summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test", "Total"],
    "Sequences": [
        len(train_df),
        len(val_df),
        len(test_df),
        len(train_df) + len(val_df) + len(test_df)
    ],
    "Unique Pedestrians": [
        len(train_peds),
        len(val_peds),
        len(test_peds),
        len(pedestrians)
    ]
})

print("=" * 55)
print("PHASE 1 - DATASET SUMMARY")
print("=" * 55)
print()
print(summary.to_string(index=False))
print()
print(f"Window size : {WINDOW}")
print(f"Stride      : {STRIDE}")
print()
print(f"Annotations CSV : {ANNOTATIONS_CSV}")
print(f"Sequences CSV   : {SEQUENCES_CSV}")
print(f"Train CSV       : {TRAIN_CSV}")
print(f"Val CSV         : {VAL_CSV}")
print(f"Test CSV        : {TEST_CSV}")

---
Phase 1 complete. The generated CSV files (`train.csv`, `val.csv`, `test.csv`) are ready for feature extraction (Phase 3) and model training (Phase 4).